← [t01 · One sky, many eyes](t01_one_sky_many_eyes.ipynb) · [Tour itinerary](README.md)
<!--nav-->

# t02 · The solar neighborhood

**The big idea:** t01 was about *pictures*; this stop is about the other thing astronomy
produces — a **census**. The [Gaia](https://www.cosmos.esa.int/web/gaia) spacecraft has spent
a decade measuring positions, motions, distances and colors for nearly **two billion stars**,
and all of it is a public database you can query like any other. In this notebook you will
run one query and hold, in a single dataframe, essentially *every star within 220
light-years* — then make the three plots that turn a star list into actual understanding:
a map, an HR diagram, and a speed distribution.

**How Gaia knows distances** (the one concept everything below rests on): **parallax**. As
Earth orbits the Sun, a nearby star appears to shift against the distant background — the
same reason your thumb jumps against the wall when you blink eyes. The shift is tiny: our
*nearest* star moves by 0.77 arcseconds, a golf ball seen from 11 km. Gaia measures shifts a
thousand times smaller. Distance in parsecs is just `1000 / parallax_in_mas` — so a cut of
`parallax >= 15` means "within 67 parsecs" (~220 light-years).

📖 *Resources:* [Parallax](https://en.wikipedia.org/wiki/Stellar_parallax) ·
[Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) ·
[primer p2](../primer/p2_stars.ipynb) (stars & the HR diagram, on a smaller sample)

In [ ]:
import os
from pathlib import Path

for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "4")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CACHE = Path("../data/cache/tour")
CACHE.mkdir(parents=True, exist_ok=True)
SAMPLE = CACHE / "gaia_67pc.parquet"

# The sample:
#   parallax >= 15            -> within 67 pc (220 ly)
#   parallax_over_error >= 10 -> distance known to better than 10%
#   ruwe < 1.4                -> astrometry not polluted by unresolved binarity
COLUMNS = ("source_id, ra, dec, parallax, pmra, pmdec, radial_velocity, "
           "phot_g_mean_mag, bp_rp")
WHERE = ("parallax_over_error >= 10 AND ruwe < 1.4 "
         "AND phot_g_mean_mag IS NOT NULL AND bp_rp IS NOT NULL")


def fetch_gaia_sample():
    """Fetch the full 67 pc sample from the ESA Gaia archive.

    Real-world plumbing, worth seeing once: the archive's *async* queue returns the
    whole result in one job but is sometimes down; the *sync* endpoint is reliable but
    capped at 2,000 rows per query. So: try async once, and on failure shard the query
    into parallax bands sized to fit under the sync cap. Volume-limited star counts
    fall as parallax^-3, so bands with equal expected counts are evenly spaced in
    parallax^-3 - and any band that still overflows just gets split again.
    """
    import time
    from astropy.table import vstack
    from astroquery.gaia import Gaia

    Gaia.ROW_LIMIT = -1

    def sync(q, tries=3):
        for attempt in range(tries):
            try:
                return Gaia.launch_job(q).get_results()
            except Exception:
                if attempt == tries - 1:
                    raise
                time.sleep(5)

    try:
        return Gaia.launch_job_async(
            f"SELECT {COLUMNS} FROM gaiadr3.gaia_source WHERE parallax >= 15 AND {WHERE}"
        ).get_results()
    except Exception as err:
        print(f"async queue unavailable ({err}); falling back to banded sync queries")

    n = int(sync(f"SELECT COUNT(*) AS n FROM gaiadr3.gaia_source "
                 f"WHERE parallax >= 15 AND {WHERE}")["n"][0])
    edges = np.linspace(15.0 ** -3, 1e-9, int(np.ceil(n / 1500)) + 1) ** (-1 / 3)
    edges[-1] = 1e6
    parts, todo = [], list(zip(edges[:-1], edges[1:]))
    while todo:
        lo, hi = todo.pop(0)
        t = sync(f"SELECT TOP 2000 {COLUMNS} FROM gaiadr3.gaia_source "
                 f"WHERE parallax >= {lo} AND parallax < {hi} AND {WHERE}")
        if len(t) == 2000:                              # cap hit: split and redo
            mid = ((lo ** -3 + hi ** -3) / 2) ** (-1 / 3)
            todo = [(lo, mid), (mid, hi)] + todo
        elif len(t):
            parts.append(t)
    full = vstack(parts)
    assert len(full) == n, "banded fetch lost rows - rerun"
    return full


if not SAMPLE.exists():
    fetch_gaia_sample().to_pandas().to_parquet(SAMPLE)
stars = pd.read_parquet(SAMPLE)

stars["dist_pc"] = 1000.0 / stars["parallax"]
print(f"{len(stars):,} stars within 67 pc pass the quality cuts")

Three cuts in that query deserve a pause, because **catalog cuts are where science silently
happens** (the same lesson as the search-design notebook, [course 06](../course/06_search_design.ipynb)):

- `parallax_over_error >= 10` — keep only stars whose *distance* is trustworthy; without it,
  the faint end fills with junk measurements.
- `ruwe < 1.4` — Gaia's "this single-star model actually fits" statistic. It quietly removes
  many **binaries**. Good for a clean diagram; terrible if binaries were what you wanted.
  Every cut is a choice about what you'll be unable to see.
- Nothing here selects on brightness — yet the sample is still incomplete: the dimmest red
  dwarfs and brown dwarfs near the 67 pc edge fall below Gaia's detection limit. Even a
  two-billion-star census has a selection function.

## 1. Sanity check: who's nearest?

If the query is right, the top of the list should be familiar names — these are the most
famous stars in the sky *because* they're nearest.

In [ ]:
nearest = stars.nlargest(8, "parallax").copy()

# Best-effort names from SIMBAD (skipped gracefully if offline). One trap worth
# meeting on day one: the nearest stars MOVE -- Barnard's Star crosses 10 arcsec of
# sky per year, so its Gaia-epoch (2016) position misses a J2000 catalog position by
# nearly 3 arcminutes. Naive cross-matching fails precisely for the most famous
# stars; rewind each position along its proper motion before matching.
names = []
try:
    import re
    from astropy.coordinates import SkyCoord
    from astropy import units as u
    from astroquery.simbad import Simbad

    for _, s in nearest.iterrows():
        years_back = 16.0                                   # Gaia DR3 epoch 2016 -> J2000
        ra_2000 = s.ra - years_back * s.pmra / 3.6e6 / np.cos(np.radians(s.dec))
        dec_2000 = s.dec - years_back * s.pmdec / 3.6e6
        res = Simbad.query_region(SkyCoord(ra_2000 * u.deg, dec_2000 * u.deg),
                                  radius=10 * u.arcsec)
        # the cone also returns the star's *planets* ("GJ 447 b") -- skip those
        found = [r["main_id"] for r in (res or [])
                 if not re.search(r" [b-h]$", r["main_id"])]
        names.append(found[0] if found else "?")
except Exception as err:
    names = ["(SIMBAD offline)"] * len(nearest)

nearest["name"] = names
nearest["dist_ly"] = nearest["dist_pc"] * 3.2616
nearest[["name", "dist_ly", "phot_g_mean_mag", "bp_rp"]].round(2)

Proxima Centauri and Barnard's Star, straight out of a database query — no star lore
required, the *data* knows. Notice their `bp_rp` colors up around 3–4: **the nearest stars
are nearly all dim red dwarfs**, invisible to the naked eye. The bright stars you see at
night are mostly rare, distant beasts; the *typical* star is one you have never seen.

Also notice who's *missing*: Sirius (8.6 ly) and α Centauri A & B never appear, because
stars that bright saturate Gaia's detectors — no clean DR3 astrometry, no row in our sample.
A flagship survey has holes at **both** ends of the brightness scale.

## 2. The map: our 220-light-year bubble

Parallax plus sky position gives full 3D coordinates. Here is the neighborhood from above —
the Sun at the center, the plane of the Galaxy as the page:

In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as u

coords = SkyCoord(ra=stars["ra"].values * u.deg, dec=stars["dec"].values * u.deg,
                  distance=stars["dist_pc"].values * u.pc)
gal = coords.galactic.cartesian   # x toward the galactic center, z out of the disk

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.scatter(gal.x.value, gal.y.value, s=1, alpha=0.25, color="#39516b")
ax.plot(0, 0, "*", color="orange", ms=16, mec="k", label="the Sun")
from matplotlib.patches import Circle
ax.add_patch(Circle((-43.5, 0), 7, fill=False, ec="crimson", lw=1.5))
ax.annotate("Hyades cluster", (-43.5, -9), ha="center", va="top", color="crimson", fontsize=9)
ax.set_xlabel("x toward the galactic center (pc)")
ax.set_ylabel("y in the direction of galactic rotation (pc)")
ax.set_title(f"{len(stars):,} stars within 67 pc, viewed from 'above' the Galaxy")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.show()

d = stars["dist_pc"]
for r in (10, 30, 67):
    n = (d <= r).sum()
    print(f"within {r:2d} pc: {n:6,} stars  ({n / (4/3*np.pi*r**3):.4f} per cubic parsec)")

Two honest features of those numbers:

- **Space is empty.** ~0.06 stars per cubic parsec means each star owns ~17 pc³ to itself —
  typical nearest-neighbor separations of a couple of parsecs, which is why stars essentially
  never collide, even when whole galaxies do.
- **Real structure is visible.** The knot circled in red is the [Hyades](https://en.wikipedia.org/wiki/Hyades_(star_cluster)) —
  the nearest star cluster, ~600 stars born together ~650 Myr ago, sitting 47 pc away exactly
  where galactic coordinates put it. You didn't plot a cluster catalog; the census just
  *contains* it.
- **The density is flat with radius — and you should interrogate that, not applaud it.** It
  says Gaia detects even faint red dwarfs out to 67 pc (true: its magnitude limit reaches
  the bottom of the main sequence at this distance). But dedicated nearby-star censuses
  count noticeably more than our 236 objects within 10 pc. The difference is the stuff our
  cuts and Gaia's limits exclude: brown dwarfs (invisible to Gaia at any radius here),
  close binaries (killed by our own `ruwe` cut), and the over-bright (Sirius). A census
  whose density looks clean can still be quietly incomplete — *ask what the instrument and
  the cuts can't see before interpreting what they show.*

## 3. The HR diagram: a census becomes physics

Plot color (temperature) against true brightness, and stars refuse to scatter randomly —
they fall onto tracks that *are* the theory of stellar structure, drawn by the data itself.
[Primer p2](../primer/p2_stars.ipynb) built this with 8,000 stars; here it is with 77,000 and
explicit quality cuts.

In [ ]:
# absolute magnitude: what G-band brightness each star would have at a standard 10 pc
stars["abs_g"] = stars["phot_g_mean_mag"] - 5 * np.log10(stars["dist_pc"] / 10)

fig, ax = plt.subplots(figsize=(8, 8.5))
h = ax.hexbin(stars["bp_rp"], stars["abs_g"], gridsize=250, bins="log", cmap="inferno")
ax.invert_yaxis()
ax.set_xlabel("Color: BP \u2212 RP (blue \u2190 hot \u00b7 cool \u2192 red)")
ax.set_ylabel("Absolute G magnitude (true brightness)")
ax.set_title("HR diagram of the solar neighborhood \u2014 every point is physics")
fig.colorbar(h, ax=ax, label="stars per bin (log)")

for x, y, text in [
    (-0.25, 4.0, "MAIN SEQUENCE\nstars fusing hydrogen;\nposition = mass"),
    (2.7, 0.8, "red clump / giants:\nold stars burning helium,\nswollen 10\u00d7"),
    (0.9, 12.8, "white dwarfs: dead cores,\nEarth-sized, slowly cooling"),
    (4.6, 9.0, "red dwarfs:\nthe silent majority"),
]:
    ax.annotate(text, (x, y), fontsize=9, ha="center", color="#1a2a3a",
                bbox=dict(boxstyle="round", fc="white", ec="gray", alpha=0.85))
plt.show()

Read it like a life-cycle diagram: a star spends ~90% of its life parked on the **main
sequence** at the spot its mass dictates, swells into the **giant** region when its core
hydrogen runs out, and ends (if sun-sized) as a **white dwarf** sliding down the cooling
track at lower left. The diagram even shows subtle structure *within* the tracks — the main
sequence is visibly thick in places (unresolved binaries that survived the RUWE cut sit
~0.75 mag above it), and the white-dwarf track splits by core composition. Professionals
mine exactly this plot, made from exactly this query.

## 4. The neighborhood is moving

Gaia measures motion too: proper motion (drift across the sky) for everything, plus radial
velocity (along the line of sight, from the Doppler shift) for the brighter stars. Convert
proper motion to km/s and the neighborhood stops being a still photograph:

In [ ]:
# tangential velocity in km/s: 4.74 is the unit conversion (mas/yr * pc -> km/s)
pm_total = np.hypot(stars["pmra"], stars["pmdec"])
stars["v_tan"] = 4.74e-3 * pm_total * stars["dist_pc"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(stars["v_tan"], bins=np.arange(0, 160, 2), color="#39516b")
ax.axvline(np.median(stars["v_tan"]), color="crimson", ls="--",
           label=f"median {np.median(stars['v_tan']):.0f} km/s")
ax.set_xlabel("Speed across the sky (km/s)")
ax.set_ylabel("Stars")
ax.set_title("How fast the neighbors move relative to us")
ax.legend()
plt.show()

fast = stars[stars["v_tan"] > 150]
print(f"stars moving faster than 150 km/s: {len(fast)}")
print("These are mostly visitors from the galactic *halo* \u2014 ancient stars on plunging")
print("orbits, just passing through the disk. Kinematics sorts stars by origin.")

A typical neighbor drifts by at ~30 km/s — about the speed Earth orbits the Sun. On top of
that, everything here orbits the galactic center together at ~230 km/s. The fast tail is a
different population entirely: **halo stars**, typically old and metal-poor, on steep orbits
that pierce the disk. That one histogram is the seed of *galactic archaeology* — using
motions as fossils of how the Galaxy assembled — which is stop t07's whole subject.

## 5. The frontier, and what's next

- **Gaia DR4 lands on 2 December 2026** — a few months from when this stop was written. It
  doubles the time baseline and releases the individual-epoch astrometry, which is expected
  to yield **~20,000 exoplanet candidates detected by the star's wobble** — potentially more
  planets from one data release than humanity has found in total to date.
- **Dark companions.** Gaia astrometry already found the first quiet (non-accreting) black
  holes in wide orbits around ordinary stars — Gaia BH1–BH3 — from stellar wobbles with no
  light source. DR4's epoch data will turn that trickle into a survey. Ironically, the
  `ruwe < 1.4` cut *we* used throws those systems away — the rejects are somebody's frontier.
- **The missing neighbors.** Gaia can't see the coolest brown dwarfs; the census within even
  10 pc is still growing (WISE found Luhman 16, the *third-closest* system, only in 2013).
  New nearest-neighbors are still discoverable, today, in public infrared data.

**Where a hobbyist fits:** the missing-neighbor hunt is genuinely amateur-accessible —
[Backyard Worlds: Planet 9](https://www.zooniverse.org/projects/marckuchner/backyard-worlds-planet-9)
volunteers have co-authored discoveries of nearby brown dwarfs by blinking WISE images. And
every M-dwarf transit target the [course](../course/README.md) taught you to vet comes from
exactly the kind of Gaia-quality-cut sample you just built.

*The Gaia sample caches to `data/cache/tour/gaia_67pc.parquet` (~5 MB); re-runs are offline
apart from the optional SIMBAD name lookup.*

---
Next stop when built: t03 · the solar system's small bodies — the census that Rubin is
currently rewriting nightly.